<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

## Theoretical Foundations
The implemented method treats neurointerventional tool detection as foreground extraction from a mostly static fluoroscopic background.

Let I_t(x, y) denote a fluoroscopy frame and B(x, y) the processed reference background. The workflow combines spatial smoothing, radiometric normalization, signed subtraction, frequency-domain high-pass filtering, morphology, and binary segmentation. The same mathematical operations are applied to every evaluation frame.


## 1. Reusable Processing Functions
The helper layer enforces one mathematical definition per operation.

Percentile saturation maps an image to the interval [0, 1] using selected lower and upper percentiles. Centralizing this transform, the loaders, the segmentation rules, the metrics, and the overlay functions prevents frame-dependent implementation drift and makes the sequence evaluation reproducible.


## 2. Data and Ground-Truth Validation
Quantitative validation requires exact correspondence between each fluoroscopy frame and its manual annotation.

The guidewire and microcatheter annotations are combined into one tool mask. Because the laboratory annotation convention represents the tool as dark pixels, the final evaluation convention remains:

- 0 = tool
- 1 = background

Consistent image dimensions are essential because subtraction and all reported metrics are pixelwise operations.


## 3. Static Background Reference
The implementation uses processed frame 201 as a static reference:

B = P(I_201)

where P denotes the common preprocessing operator. This assumes that the anatomy is largely static and that later deviations are dominated by moving tools, noise, and residual intensity variation.


## 4. Intermediate Processing Pipeline
For an evaluation frame, preprocessing first applies Gaussian smoothing with sigma = 1 and then percentile normalization.

The signed residual is computed as:

D_t = -(P(I_t) - B)

so darker moving structures relative to the background become positive responses.

A centered 2-D Fourier transform is then filtered by a Gaussian high-pass transfer function. Positive spectral responses are morphologically dilated and normalized to form the segmentation feature image.


## 5. Segmentation Strategy Generation
The fixed-threshold strategy applies a deterministic decision at T = 0.10.

Otsu thresholding estimates a scalar threshold by minimizing within-class variance in the feature histogram.

The EM/GMM strategy models feature values with two Gaussian components. The component with the larger mean is interpreted as the higher-response tool class.

All three strategies use the same final morphological opening before conversion to the required 0=tool, 1=background mask convention.


## 6. Representative Strategy Comparison
A single representative frame provides a controlled visual comparison because the input, ground truth, feature image, and postprocessing are shared.

The semantic overlay distinguishes three important regions: correct tool overlap, annotated tool missed by the prediction, and predicted tool unsupported by the annotation. This complements scalar metrics by showing where errors occur spatially.


## 7. Sequence-Level Evaluation
For each frame and strategy, the implementation reports pixel-domain error measures in the 0–255 mask domain.

SAD is the mean absolute pixel difference. MSE is the mean squared pixel difference. PSNR expresses the error on a logarithmic scale relative to a peak value of 255.

Dice and IoU are additionally computed after treating the tool pixels as the positive set, providing overlap-oriented measures that are useful for the sparse device class.


## 8. Aggregate Metrics and Strategy Selection
Aggregating over the sequence reduces dependence on a single favorable or difficult frame.

For every metric, the mean summarizes central performance and the standard deviation measures temporal variability. The executable notebook retains the strategy with the lowest mean MSE.


## 9. Temporal Metric Curves
Temporal metric curves reveal whether errors remain stable across the sequence.

A method can have an acceptable global mean while failing strongly on one frame. Plotting SAD, MSE, and PSNR against frame number therefore exposes frame-specific degradation that aggregate values alone cannot show.


## 10. Strategy Summary
The aggregate strategy summary places different metric families side by side.

MSE and PSNR measure pixel-domain agreement, whereas Dice measures overlap of the sparse tool class. Because their scales and optimization directions differ, the chart is interpreted as a multi-view diagnostic rather than as one universal score.


## 11. Final Pipeline Assembly
The final pipeline packages the retained sequence into one reusable inference function:

input frame -> preprocessing -> signed background residual -> Gaussian spectral high-pass filtering -> morphology -> retained segmentation strategy -> finalized binary mask.

This prevents the final visualization stage from diverging from the quantitatively evaluated method.


## 12. Final Guidance Gallery
The guidance overlay converts the binary prediction into a visually interpretable output.

The fluoroscopy frame is contrast-enhanced, converted to RGB, and predicted tool pixels are emphasized in green. This does not change the segmentation result; it only makes the detected tool easier to inspect in anatomical context.


## 13. Final Qualitative Validation
Final qualitative validation compares three representations side by side: final prediction, manual annotation, and semantic error overlay.

Using frames 211, 251, and 291 samples the beginning, middle, and end of the evaluation sequence so that the retained method is checked beyond the single representative development frame.


## 14. Export Results and Output Inventory
Saving tables and figures is part of reproducibility.

The detailed CSV preserves every frame-strategy measurement, while the summary CSV preserves the aggregated comparison. The printed figure inventory provides a direct execution check that the expected visual diagnostics were generated.


## Technical Synthesis
The implemented background-subtraction model is one reproducible chain:

validated sequence -> processed frame-201 background -> Gaussian + percentile preprocessing -> signed subtraction -> Gaussian spectral high-pass filtering -> morphological feature refinement -> segmentation comparison -> retained final strategy.

Its strength is not a single operation but the fact that every candidate segmentation method is evaluated on the same feature representation over the same sequence.


## Scope and Limitations

### Included
- static first-frame background reference;
- Gaussian spatial filtering;
- percentile-based intensity normalization;
- Fourier-domain Gaussian high-pass filtering;
- dilation and opening;
- fixed threshold, Otsu, and two-component EM/GMM segmentation;
- SAD, MSE, PSNR, Dice, and IoU evaluation;
- guidance and validation overlays.

### Limitations
- the background model is fixed to frame 201 rather than estimated adaptively;
- the method assumes spatially registered anatomy;
- fixed preprocessing parameters may not generalize to another acquisition protocol;
- the tool class is extremely sparse, so pixel-domain metrics are influenced by the dominant background;
- no learned temporal or deep model is used.
